# LeetCode 1975 — Maximum Matrix Sum bằng Python

Link đề: https://leetcode.com/problems/maximum-matrix-sum/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maxMatrixSum(self, matrix: List[List[int]]) -> int:
        ...
```

Notebook đi từ **BFS trên mọi cấu hình dấu** (vét cạn với ma trận nhỏ) → tìm **bất biến chẵn lẻ** của số phần tử âm → **công thức một lượt** `Σ|a| − 2·min|a|` khi số âm lẻ → **mô phỏng xây dựng** dãy thao tác thật theo đường **rắn bò (snake)** để thấy công thức đạt được.

## 1. Mục tiêu học

1. Nhận ra **bất biến**: mỗi thao tác đổi dấu đúng hai phần tử ⇒ **tính chẵn lẻ** của số phần tử âm không đổi (khi không có số 0).
2. Hiểu rằng có thể "di chuyển" dấu âm dọc theo các ô kề nhau ⇒ hai dấu âm bất kỳ đều triệt tiêu được.
3. Suy ra công thức: tổng trị tuyệt đối, trừ `2·min|a|` nếu số âm lẻ (và không cần trừ nếu có số 0).
4. Cài đặt một lượt `O(n²)`, và một bản **dựng lời giải** (in ra dãy thao tác) để kiểm chứng.
5. Kiểm thử bằng BFS vét cạn trên ma trận `2×2`, `2×3`, `3×3`.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `n × n`. Được thực hiện bao nhiêu lần tùy ý thao tác: chọn **hai phần tử kề nhau** (chung cạnh) và **nhân cả hai với −1**.

Mục tiêu: **tổng lớn nhất** của các phần tử. Trả về tổng đó.

**Ví dụ 1:** `matrix = [[1,-1],[-1,1]]` → `4` (đổi dấu hàng đầu, rồi cột đầu).

**Ví dụ 2:** `matrix = [[1,2,3],[-1,-2,-3],[1,2,3]]` → `16` (đổi dấu hai phần tử cuối của hàng giữa; còn lại `-1` nên tổng `18 − 2 = 16`).

**Ràng buộc:** `2 ≤ n ≤ 250`, `−10⁵ ≤ matrix[i][j] ≤ 10⁵`.

In [1]:
import random
import time
from collections import deque
from itertools import product
from typing import List

examples = [([[1, -1], [-1, 1]], 4), ([[1, 2, 3], [-1, -2, -3], [1, 2, 3]], 16)]

def flip(mat, a, b):
    """Thao tác: nhân hai ô kề a, b với -1 (trả ma trận mới)."""
    (r1, c1), (r2, c2) = a, b
    assert abs(r1 - r2) + abs(c1 - c2) == 1, "phải kề nhau"
    m = [row[:] for row in mat]
    m[r1][c1] *= -1; m[r2][c2] *= -1
    return m

m = examples[0][0]
m = flip(m, (0, 0), (0, 1)); print("sau đổi hàng đầu:", m)
m = flip(m, (0, 0), (1, 0)); print("sau đổi cột đầu :", m, "tổng =", sum(map(sum, m)))

sau đổi hàng đầu: [[-1, 1], [-1, 1]]
sau đổi cột đầu : [[1, 1], [1, 1]] tổng = 4


## 3. Kiến thức nền

### 3.1 Ô kề trong lưới

Ô `(r, c)` kề với `(r±1, c)` và `(r, c±1)` (trong biên).

In [2]:
def neighbors(r, c, R, C):
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < R and 0 <= nc < C:
            yield nr, nc

print(list(neighbors(0, 0, 3, 3)), list(neighbors(1, 1, 3, 3)))

[(1, 0), (0, 1)] [(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.2 Bất biến chẵn lẻ

Đổi dấu hai số khác 0: số lượng số âm thay đổi `+2`, `0` hoặc `−2` ⇒ **chẵn lẻ không đổi**. Nếu có số 0, đổi dấu 0 không đổi gì ⇒ có thể "hút" một dấu âm vào số 0.

In [3]:
def neg_count(mat):
    return sum(x < 0 for row in mat for x in row)

m = [[1, -2], [3, 4]]
print("số âm ban đầu:", neg_count(m))
for a, b in [((0, 0), (0, 1)), ((0, 1), (1, 1)), ((0, 0), (1, 0))]:
    print(f"đổi {a},{b}: số âm = {neg_count(flip(m, a, b))}")

số âm ban đầu: 1
đổi (0, 0),(0, 1): số âm = 1
đổi (0, 1),(1, 1): số âm = 1
đổi (0, 0),(1, 0): số âm = 3


### 3.3 Đường rắn bò (snake order)

Duyệt hàng 0 trái → phải, hàng 1 phải → trái, hàng 2 trái → phải… Hai ô liên tiếp trên đường này **luôn kề nhau** ⇒ có thể "đẩy" dấu âm dọc theo đường.

In [4]:
def snake(R, C):
    out = []
    for r in range(R):
        cols = range(C) if r % 2 == 0 else range(C - 1, -1, -1)
        out += [(r, c) for c in cols]
    return out

path = snake(3, 3)
print(path)
print("mọi cặp liên tiếp đều kề:", all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(path, path[1:])))

[(0, 0), (0, 1), (0, 2), (1, 2), (1, 1), (1, 0), (2, 0), (2, 1), (2, 2)]
mọi cặp liên tiếp đều kề: True


### 3.4 BFS trên cấu hình dấu

Một cấu hình là vector dấu `±` của các ô; với ma trận nhỏ (`≤ 9` ô) có `≤ 512` cấu hình — BFS duyệt hết dễ dàng.

In [5]:
print("số cấu hình dấu của ma trận 3x3:", 2 ** 9)

số cấu hình dấu của ma trận 3x3: 512


## 4. Ý tưởng nền tảng / trực giác

Hãy để ý **hai dấu âm ở xa nhau** vẫn triệt tiêu được: đổi dấu cặp `(âm, kề)` làm dấu âm "chạy" sang ô kề (ô cũ thành dương, ô kề đổi dấu). Lặp lại, dấu âm chạy dọc một đường cho tới khi gặp dấu âm khác ⇒ cả hai thành dương.

⇒ Mọi dấu âm ghép cặp được. Nếu số âm **chẵn**: tất cả thành dương ⇒ tổng `Σ|a|`. Nếu **lẻ**: còn đúng một dấu âm, đặt nó ở ô có **|a| nhỏ nhất** ⇒ tổng `Σ|a| − 2·min|a|`. (Nếu có số `0`, `min|a| = 0` nên công thức vẫn đúng.)

Tính tay ví dụ 2: `Σ|a| = 6 + 6 + 6 = 18`; có 3 số âm (lẻ); `min|a| = 1` ⇒ `18 − 2 = 16` ✔.

In [6]:
def move_negative_demo():
    m = [[-5, 1, 1], [1, 1, 1], [1, 1, -7]]
    print("ban đầu:", m, "tổng", sum(map(sum, m)))
    path = snake(3, 3)
    for a, b in zip(path, path[1:]):
        if m[a[0]][a[1]] < 0:
            m = flip(m, a, b)
            print(f"  đổi {a},{b} -> {m}")
    print("cuối:", m, "tổng", sum(map(sum, m)))

move_negative_demo()

ban đầu: [[-5, 1, 1], [1, 1, 1], [1, 1, -7]] tổng -5
  đổi (0, 0),(0, 1) -> [[5, -1, 1], [1, 1, 1], [1, 1, -7]]
  đổi (0, 1),(0, 2) -> [[5, 1, -1], [1, 1, 1], [1, 1, -7]]
  đổi (0, 2),(1, 2) -> [[5, 1, 1], [1, 1, -1], [1, 1, -7]]
  đổi (1, 2),(1, 1) -> [[5, 1, 1], [1, -1, 1], [1, 1, -7]]
  đổi (1, 1),(1, 0) -> [[5, 1, 1], [-1, 1, 1], [1, 1, -7]]
  đổi (1, 0),(2, 0) -> [[5, 1, 1], [1, 1, 1], [-1, 1, -7]]
  đổi (2, 0),(2, 1) -> [[5, 1, 1], [1, 1, 1], [1, -1, -7]]
  đổi (2, 1),(2, 2) -> [[5, 1, 1], [1, 1, 1], [1, 1, 7]]
cuối: [[5, 1, 1], [1, 1, 1], [1, 1, 7]] tổng 19


## 5. Cách 1 — Brute Force / baseline: BFS trên mọi cấu hình dấu

In [7]:
def max_sum_bfs(matrix):
    R, C = len(matrix), len(matrix[0])
    cells = [(r, c) for r in range(R) for c in range(C)]
    idx = {cell: k for k, cell in enumerate(cells)}
    edges = [(idx[a], idx[b]) for a in cells for b in neighbors(*a, R, C) if idx[a] < idx[b]]
    start = 0                                    # bitmask: bit k = 1 nghĩa là ô k bị đổi dấu
    seen = {start}; q = deque([start])
    while q:
        s = q.popleft()
        for i, j in edges:
            t = s ^ (1 << i) ^ (1 << j)
            if t not in seen:
                seen.add(t); q.append(t)
    best = None
    for s in seen:
        total = sum(-matrix[r][c] if s >> idx[(r, c)] & 1 else matrix[r][c] for r, c in cells)
        best = total if best is None else max(best, total)
    return best

for mat, exp in examples:
    print(max_sum_bfs(mat), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
16 (kỳ vọng 16 )


**Độ phức tạp:** `O(2^(n²) · n²)` — chỉ cho `n² ≤ 12` ô. Với `n = 250` bất khả thi.

**Nút thắt:** BFS cho thấy tập đạt được là **mọi** mặt nạ có **số bit chẵn** (in ở ô dưới) ⇒ không cần duyệt: chỉ cần công thức.

In [8]:
def reachable_masks(R, C):
    cells = [(r, c) for r in range(R) for c in range(C)]
    idx = {cell: k for k, cell in enumerate(cells)}
    edges = [(idx[a], idx[b]) for a in cells for b in neighbors(*a, R, C) if idx[a] < idx[b]]
    seen = {0}; q = deque([0])
    while q:
        s = q.popleft()
        for i, j in edges:
            t = s ^ (1 << i) ^ (1 << j)
            if t not in seen:
                seen.add(t); q.append(t)
    return seen

for R, C in [(2, 2), (2, 3), (3, 3)]:
    S = reachable_masks(R, C)
    print(f"{R}x{C}: đạt được {len(S)} / {2 ** (R * C)} mặt nạ; tất cả đều có số bit chẵn: {all(bin(s).count('1') % 2 == 0 for s in S)}")

2x2: đạt được 8 / 16 mặt nạ; tất cả đều có số bit chẵn: True
2x3: đạt được 32 / 64 mặt nạ; tất cả đều có số bit chẵn: True
3x3: đạt được 256 / 512 mặt nạ; tất cả đều có số bit chẵn: True


## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến:** số ô bị đổi dấu (so với ban đầu) luôn **chẵn**.

**Đủ:** mọi tập ô có kích thước chẵn đều đạt được (ghép cặp, đẩy dấu dọc đường rắn bò). Mục 5 kiểm chứng bằng BFS.

**Tối ưu:** muốn mọi số âm thành dương. Gọi `k` = số phần tử âm.
- **Nếu** `k` chẵn **thì** đổi đúng các số âm ⇒ tổng `Σ|a|`.
- **Nếu** `k` lẻ **thì** phải đổi một tập chẵn; tốt nhất là đổi `k` số âm và **thêm** (hoặc bớt) đúng một ô có `|a|` nhỏ nhất ⇒ tổng `Σ|a| − 2·min|a|`.
- **Nếu** có số `0`: `min|a| = 0` ⇒ hai trường hợp như nhau.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Công thức một lượt (lời giải chính)

In [9]:
def max_sum_formula(matrix):
    total = 0
    neg = 0
    mn = float('inf')
    for row in matrix:
        for x in row:
            if x < 0:
                neg += 1
            ax = abs(x)
            total += ax
            if ax < mn:
                mn = ax
    return total if neg % 2 == 0 else total - 2 * mn

for mat, exp in examples:
    print(max_sum_formula(mat), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
16 (kỳ vọng 16 )


**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ.

### 7.2 Cách 3 — Dựng lời giải: đẩy dấu âm theo đường rắn bò

Cách này **thực sự thực hiện** các thao tác: duyệt đường rắn bò, gặp ô âm thì đổi dấu nó cùng ô kế tiếp trên đường. Kết thúc, mọi ô dương trừ có thể ô cuối. Nếu ô cuối âm, "đẩy" dấu âm ngược lại dọc đường tới ô có `|a|` nhỏ nhất. Trả về tổng **và** danh sách thao tác (để kiểm chứng).

In [10]:
def max_sum_constructive(matrix):
    R, C = len(matrix), len(matrix[0])
    m = [row[:] for row in matrix]
    path = snake(R, C)
    ops = []
    def do(a, b):
        m[a[0]][a[1]] *= -1; m[b[0]][b[1]] *= -1; ops.append((a, b))
    for a, b in zip(path, path[1:]):
        if m[a[0]][a[1]] < 0:
            do(a, b)
    last = path[-1]
    if m[last[0]][last[1]] < 0:
        # chọn ô có |a| nhỏ nhất, đẩy dấu âm từ cuối đường lùi về đó
        target = min(range(len(path)), key=lambda k: (abs(m[path[k][0]][path[k][1]]), -k))   # |a| nhỏ nhất, gần cuối nhất
        for k in range(len(path) - 1, target, -1):
            do(path[k], path[k - 1])
    return sum(map(sum, m)), ops

for mat, exp in examples:
    total, ops = max_sum_constructive(mat)
    print(total, "(kỳ vọng", exp, ") | số thao tác:", len(ops))

4 (kỳ vọng 4 ) | số thao tác: 2
16 (kỳ vọng 16 ) | số thao tác: 6


Khi đẩy dấu âm ngược về `target`, mỗi thao tác `(path[k], path[k−1])` làm ô `k` dương lại và ô `k−1` âm; các ô trên đường từ `target+1` tới cuối trở về dương. **Độ phức tạp:** `O(n²)` thao tác, `O(n²)` thời gian.

### 7.3 Cách 4 — Sắp xếp trị tuyệt đối (biến thể của công thức)

Trải phẳng ma trận, sắp các trị tuyệt đối; nếu số âm lẻ thì phần tử nhỏ nhất (đầu dãy) mang dấu âm. Cách này chậm hơn (`O(n² log n)`) nhưng dễ mở rộng nếu đề hỏi "tổng lớn nhất khi phải để lại đúng `t` số âm" (lấy `t` trị nhỏ nhất).

In [11]:
def max_sum_sorted(matrix):
    vals = [x for row in matrix for x in row]
    neg = sum(x < 0 for x in vals)
    abs_sorted = sorted(abs(x) for x in vals)
    if neg % 2 == 1:
        abs_sorted[0] = -abs_sorted[0]
    return sum(abs_sorted)

for mat, exp in examples:
    print(max_sum_sorted(mat), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
16 (kỳ vọng 16 )


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2:

| ô | x | âm? | |x| | tổng |x| | min |x| | số âm |
|---|---|---|---|---|---|---|
| (0,0) | 1 | | 1 | 1 | 1 | 0 |
| (1,0) | −1 | ✔ | 1 | 7 | 1 | 1 |
| (1,1) | −2 | ✔ | 2 | 9 | 1 | 2 |
| (1,2) | −3 | ✔ | 3 | 12 | 1 | 3 |
| … | | | | 18 | 1 | 3 (lẻ) ⇒ `18 − 2 = 16` |

In [12]:
def trace_formula(matrix):
    total = neg = 0; mn = float('inf')
    for r, row in enumerate(matrix):
        for c, x in enumerate(row):
            neg += x < 0; total += abs(x); mn = min(mn, abs(x))
            print(f"  ô ({r},{c}) = {x:>3}: tổng|a| = {total:>3}, min|a| = {mn}, số âm = {neg}")
    ans = total if neg % 2 == 0 else total - 2 * mn
    print(f"  số âm {'chẵn' if neg % 2 == 0 else 'lẻ'} -> đáp án {ans}")
    return ans

trace_formula([[1, 2, 3], [-1, -2, -3], [1, 2, 3]])

  ô (0,0) =   1: tổng|a| =   1, min|a| = 1, số âm = 0
  ô (0,1) =   2: tổng|a| =   3, min|a| = 1, số âm = 0
  ô (0,2) =   3: tổng|a| =   6, min|a| = 1, số âm = 0
  ô (1,0) =  -1: tổng|a| =   7, min|a| = 1, số âm = 1
  ô (1,1) =  -2: tổng|a| =   9, min|a| = 1, số âm = 2
  ô (1,2) =  -3: tổng|a| =  12, min|a| = 1, số âm = 3
  ô (2,0) =   1: tổng|a| =  13, min|a| = 1, số âm = 3
  ô (2,1) =   2: tổng|a| =  15, min|a| = 1, số âm = 3
  ô (2,2) =   3: tổng|a| =  18, min|a| = 1, số âm = 3
  số âm lẻ -> đáp án 16


16

In [13]:
def trace_constructive(matrix):
    total, ops = max_sum_constructive(matrix)
    m = [row[:] for row in matrix]
    print("  ban đầu:", m)
    for a, b in ops:
        m = flip(m, a, b)
        print(f"  đổi {a} & {b} -> {m}")
    print("  tổng cuối:", total)

trace_constructive([[1, 2, 3], [-1, -2, -3], [1, 2, 3]])

  ban đầu: [[1, 2, 3], [-1, -2, -3], [1, 2, 3]]
  đổi (1, 2) & (1, 1) -> [[1, 2, 3], [-1, 2, 3], [1, 2, 3]]
  đổi (1, 0) & (2, 0) -> [[1, 2, 3], [1, 2, 3], [-1, 2, 3]]
  đổi (2, 0) & (2, 1) -> [[1, 2, 3], [1, 2, 3], [1, -2, 3]]
  đổi (2, 1) & (2, 2) -> [[1, 2, 3], [1, 2, 3], [1, 2, -3]]
  đổi (2, 2) & (2, 1) -> [[1, 2, 3], [1, 2, 3], [1, -2, 3]]
  đổi (2, 1) & (2, 0) -> [[1, 2, 3], [1, 2, 3], [-1, 2, 3]]
  tổng cuối: 16


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def maxMatrixSum(self, matrix: List[List[int]]) -> int:
        total = 0
        neg = 0
        mn = float('inf')
        for row in matrix:
            for x in row:
                if x < 0:
                    neg += 1
                ax = -x if x < 0 else x
                total += ax
                if ax < mn:
                    mn = ax
        return total if neg % 2 == 0 else total - 2 * mn

for mat, exp in examples:
    assert Solution().maxMatrixSum(mat) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `F` là tập ô bị đổi dấu sau cùng (so với ban đầu). Tổng cuối `= Σ_{ô ∉ F} a − Σ_{ô ∈ F} a`.

**Bất biến.** Mỗi thao tác đảo trạng thái "thuộc `F`" của đúng hai ô ⇒ `|F|` luôn **chẵn**.

**Mọi `F` chẵn đều đạt được.** Liệt kê `F` theo thứ tự đường rắn bò, ghép thành cặp liên tiếp `(u₁, v₁), (u₂, v₂), …`; với mỗi cặp, áp thao tác lên mọi cặp ô liên tiếp trên đường từ `u` tới `v` — các ô ở giữa bị đảo hai lần (không đổi), chỉ `u`, `v` bị đảo một lần.

**Tối ưu.** Ta muốn `F` = tập số âm `N` (khi đó tổng `= Σ|a|`). Nếu `|N|` chẵn — đạt được. Nếu lẻ, mọi `F` hợp lệ khác `N` ít nhất một ô; đổi khác một ô `x` làm mất `2|x|` so với `Σ|a|` ⇒ tốt nhất mất `2·min|a|`, và đạt được bằng `F = N △ {ô có |a| nhỏ nhất}` (kích thước chẵn).

Mục 5 xác nhận bằng BFS rằng tập đạt được đúng là mọi `F` có kích thước chẵn (với các lưới nhỏ).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS cấu hình dấu | `O(2^(n²) · n²)` | `O(2^(n²))` |
| 2. Công thức một lượt | `O(n²)` | `O(1)` |
| 3. Dựng lời giải (snake) | `O(n²)` | `O(n²)` |
| 4. Sắp xếp trị tuyệt đối | `O(n² log n)` | `O(n²)` |

Tổng tới `250² · 10⁵ ≈ 6.25·10⁹` — vượt `int` 32-bit, cần `long` trong C++/Java.

In [15]:
rnd = random.Random(1975)
big = [[rnd.randint(-10**5, 10**5) for _ in range(250)] for _ in range(250)]
for name, fn in [("formula", max_sum_formula), ("sorted", max_sum_sorted), ("constructive", lambda m: max_sum_constructive(m)[0]), ("Solution", Solution().maxMatrixSum)]:
    t0 = time.perf_counter(); r = fn(big); dt = time.perf_counter() - t0
    print(f"{name:<13}: {r}  {dt * 1000:7.1f} ms")

formula      : 3125934283      4.1 ms
sorted       : 3125934283     14.3 ms
constructive : 3125934283     18.4 ms
Solution     : 3125934283      4.1 ms


## 12. Bộ test

In [16]:
ALL = {
    "formula": max_sum_formula,
    "sorted": max_sum_sorted,
    "constructive": lambda m: max_sum_constructive(m)[0],
    "Solution": lambda m: Solution().maxMatrixSum(m),
}

def check(mat, exp=None, brute=True):
    snap = [row[:] for row in mat]
    ref = max_sum_bfs(mat) if brute else max_sum_formula(mat)
    for name, fn in ALL.items():
        assert fn(mat) == ref, (name, mat)
        assert mat == snap, f"{name} đã sửa input"
    if exp is not None:
        assert ref == exp
    return ref

for mat, exp in examples:
    check(mat, exp)
print("Ví dụ đề: OK")

check([[-1, -1], [-1, -1]], 4)            # 4 số âm (chẵn)
check([[-1, -1], [-1, 1]], 2)             # 3 số âm, min 1
check([[0, -5], [-3, 2]], 10)             # có số 0: hút dấu âm
check([[-7, 0], [0, 0]], 7)
check([[-10**5, -10**5], [-10**5, 2]], 3 * 10**5 - 2)
print("Edge case: OK")

rnd = random.Random(2003)
for _ in range(600):
    R, C = rnd.choice([(2, 2), (2, 3), (3, 2), (3, 3)])
    mat = [[rnd.randint(-5, 5) for _ in range(C)] for _ in range(R)]
    check(mat)
print("600 ma trận nhỏ so với BFS: OK")

for _ in range(30):
    n = rnd.randint(2, 60)
    check([[rnd.randint(-10**5, 10**5) for _ in range(n)] for _ in range(n)], brute=False)
print("30 ma trận cỡ vừa: các cách khớp nhau: OK")

Ví dụ đề: OK
Edge case: OK


600 ma trận nhỏ so với BFS: OK


30 ma trận cỡ vừa: các cách khớp nhau: OK


Trong `check` có ma trận `2×3`, `3×2` — không vuông như đề nhưng công thức vẫn đúng; giữ để tăng độ phủ của kiểm thử. Các edge case trên là **tự dựng**, kỳ vọng đã được BFS xác nhận (đối chiếu lại đề gốc nếu muốn).

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đổi dấu các cặp âm **kề nhau**

In [17]:
def max_sum_bug_adjacent_only(matrix):
    m = [row[:] for row in matrix]; R, C = len(m), len(m[0])
    changed = True
    while changed:
        changed = False
        for r in range(R):
            for c in range(C):
                for nr, nc in neighbors(r, c, R, C):
                    if m[r][c] < 0 and m[nr][nc] < 0:      # SAI: chỉ ghép hai số âm cạnh nhau
                        m[r][c] *= -1; m[nr][nc] *= -1; changed = True
    return sum(map(sum, m))

mat = [[-1, 2], [3, -4]]
print("sai :", max_sum_bug_adjacent_only(mat), "| đúng:", Solution().maxMatrixSum(mat))

sai : 0 | đúng: 10


**Vì sao sai:** hai dấu âm ở xa vẫn triệt tiêu được bằng cách "đẩy" dấu qua các ô dương ở giữa. **Sửa:** chỉ cần đếm chẵn lẻ (công thức).

### Lỗi 2 — Khi số âm lẻ, trừ `2 ×` **số âm nhỏ nhất** thay vì phần tử có `|a|` nhỏ nhất

In [18]:
def max_sum_bug_min_negative(matrix):
    vals = [x for row in matrix for x in row]
    neg = [x for x in vals if x < 0]
    total = sum(abs(x) for x in vals)
    return total if len(neg) % 2 == 0 else total - 2 * min(abs(x) for x in neg)   # SAI

mat = [[-5, 1], [2, 3]]
print("sai :", max_sum_bug_min_negative(mat), "| đúng:", Solution().maxMatrixSum(mat))

sai : 1 | đúng: 9


**Vì sao sai:** dấu âm còn lại có thể được **đẩy sang** một số **dương** nhỏ hơn (ở đây `1`), không nhất thiết nằm trên một số âm. **Sửa:** `min` trên **mọi** `|a|`.

### Lỗi 3 — Quên trường hợp có số 0

In [19]:
def max_sum_bug_zero(matrix):
    vals = [x for row in matrix for x in row]
    neg = sum(x < 0 for x in vals)
    nonzero_min = min(abs(x) for x in vals if x != 0)       # SAI: bỏ qua số 0
    total = sum(abs(x) for x in vals)
    return total if neg % 2 == 0 else total - 2 * nonzero_min

mat = [[0, -5], [-3, -2]]
print("sai :", max_sum_bug_zero(mat), "| đúng:", Solution().maxMatrixSum(mat))

sai : 6 | đúng: 10


**Vì sao sai:** số 0 "hút" được dấu âm mà không mất gì (`−0 = 0`). **Sửa:** tính `min|a|` trên mọi phần tử, kể cả 0.

### Lỗi 4 — Đếm số âm sau khi đã lấy trị tuyệt đối

In [20]:
def max_sum_bug_order(matrix):
    vals = [abs(x) for row in matrix for x in row]          # SAI: mất thông tin dấu
    neg = sum(x < 0 for x in vals)
    return sum(vals) if neg % 2 == 0 else sum(vals) - 2 * min(vals)

mat = [[1, 2, 3], [-1, -2, -3], [1, 2, 3]]
print("sai :", max_sum_bug_order(mat), "| đúng:", Solution().maxMatrixSum(mat))

sai : 18 | đúng: 16


**Sửa:** đếm `x < 0` **trước** (hoặc song song) với việc lấy trị tuyệt đối.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thao tác "đổi dấu **hai** phần tử" ⇒ tìm **bất biến chẵn lẻ** của số phần tử âm.
- Đồ thị các ô **liên thông** (lưới) ⇒ có thể "di chuyển" một dấu âm tới bất kỳ đâu ⇒ chỉ chẵn lẻ quan trọng.
- Số lẻ dấu âm ⇒ để lại trên phần tử `|a|` nhỏ nhất (toàn bộ ma trận, không chỉ số âm).
- Kiểm tập trạng thái đạt được bằng BFS trên ví dụ nhỏ trước khi tin công thức.

## 15. Biến thể của bài

1. **Mảng 1 chiều, đổi dấu hai phần tử kề** — giống hệt (đường thẳng cũng liên thông).
2. **Đổi dấu hai phần tử bất kỳ** (không cần kề): cùng đáp án.
3. **Đổi dấu đúng `k` lần** (LeetCode 1005 — Maximize Sum Of Array After K Negations, đổi dấu **một** phần tử mỗi lần): chẵn lẻ của `k` còn lại quyết định.
4. **Đồ thị không liên thông** (ví dụ chỉ được đổi theo một số cặp cho trước): xét chẵn lẻ **trên từng thành phần liên thông**.
5. **Đổi dấu cả một hàng hoặc cả một cột** (LeetCode 861-like): bài khác hẳn (tham lam theo hàng/cột).

In [21]:
def max_sum_after_k_negations(nums, k):
    """LeetCode 1005: mỗi lần đổi dấu MỘT phần tử, đúng k lần."""
    a = sorted(nums)
    for i in range(len(a)):
        if k > 0 and a[i] < 0:
            a[i] = -a[i]; k -= 1
    total = sum(a)
    if k % 2 == 1:
        total -= 2 * min(a)
    return total

def max_sum_components(values, pairs):
    """Biến thể 4: đổi dấu hai đầu của một cạnh trong 'pairs'; xét chẵn lẻ từng thành phần."""
    n = len(values); parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for u, v in pairs:
        parent[find(u)] = find(v)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(values[i])
    ans = 0
    for g in groups.values():
        if len(g) == 1:
            ans += g[0]; continue
        neg = sum(x < 0 for x in g); s = sum(abs(x) for x in g)
        ans += s if neg % 2 == 0 else s - 2 * min(abs(x) for x in g)
    return ans

print("1005 [4,2,3], k=1 ->", max_sum_after_k_negations([4, 2, 3], 1), "| [3,-1,0,2], k=3 ->", max_sum_after_k_negations([3, -1, 0, 2], 3))
print("thành phần {0,1} và {2}: [-3, 2, -1] ->", max_sum_components([-3, 2, -1], [(0, 1)]))

1005 [4,2,3], k=1 -> 5 | [3,-1,0,2], k=3 -> 6
thành phần {0,1} và {2}: [-3, 2, -1] -> 0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | mọi cấu hình dấu | `O(2^(n²) n²)` | lớn | kiểm chứng lưới nhỏ |
| 2. Công thức | `Σ|a|` trừ `2·min|a|` nếu số âm lẻ | `O(n²)` | `O(1)` | **nộp bài** |
| 3. Dựng lời giải | đẩy dấu theo đường rắn bò | `O(n²)` | `O(n²)` | cần in thao tác |
| 4. Sắp xếp | trị tuyệt đối tăng dần | `O(n² log n)` | `O(n²)` | mở rộng "để lại t số âm" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().maxMatrixSum([[-1, 0], [-1, -1]])` và `([[2, -3], [4, -5]])`?

**Bài 2 (tính tay):** Với `[[-4, 3], [2, -1]]`, tính `Σ|a|`, số âm, và đáp án; viết một dãy thao tác đạt đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `final_matrix(matrix)` trả về **ma trận cuối** đạt tổng lớn nhất (dùng cách 3), kiểm tổng bằng `Solution`.

**Bài 4 (tự viết lại):** Viết lại cách 3 cho **lưới chữ nhật** `R × C` bất kỳ và **đếm số thao tác** đã dùng; chứng tỏ số thao tác `≤ 2·R·C`.

**Bài 5 (mini challenge):** Biến thể: chỉ được đổi dấu hai ô kề **cùng hàng** (không theo cột). Viết `max_sum_row_only(matrix)` (xét từng hàng độc lập) và kiểm bằng BFS trên lưới nhỏ.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def final_matrix(matrix):
    # TODO: chạy max_sum_constructive, áp các thao tác lên bản sao, trả ma trận
    pass

# Bài 4 — TODO
def constructive_rect(matrix):
    # TODO: snake(R, C) cho lưới chữ nhật; trả (tổng, số thao tác)
    pass

# Bài 5 — TODO
def max_sum_row_only(matrix):
    # TODO: mỗi hàng là một thành phần liên thông (đường thẳng): áp công thức cho từng hàng
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Ma trận đầu có số 0.
- Mức 2: `Σ|a| = 3`, có 0 ⇒ đáp án 3.
- Mức 3: Ma trận hai: `Σ|a| = 14`, 2 số âm (chẵn) ⇒ 14.

**Bài 2**
- Mức 1: `Σ|a| = 10`, 2 số âm.
- Mức 2: Đáp án 10.
- Mức 3: Đổi `(0,0)&(0,1)` rồi `(0,1)&(1,1)`.

**Bài 3**
- Mức 1: Dùng danh sách `ops`.
- Mức 2: Áp `flip` lần lượt.
- Mức 3: Tổng ma trận cuối = `Solution()`.

**Bài 4**
- Mức 1: `snake` đã hỗ trợ `R × C`.
- Mức 2: Lượt đẩy tiến tối đa `RC − 1` thao tác, lượt lùi tối đa `RC − 1`.
- Mức 3: Kiểm `len(ops) ≤ 2RC`.

**Bài 5**
- Mức 1: Không có thao tác nối hai hàng ⇒ hàng độc lập.
- Mức 2: Mỗi hàng: công thức chẵn lẻ riêng.
- Mức 3: BFS với cạnh chỉ trong hàng để kiểm.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = (Solution().maxMatrixSum([[-1, 0], [-1, -1]]), Solution().maxMatrixSum([[2, -3], [4, -5]]))
print("Bài 1:", r1)
assert r1 == (3, 14)

# Bài 2
m2 = [[-4, 3], [2, -1]]
print("Bài 2: Σ|a| =", sum(abs(x) for row in m2 for x in row), "| số âm =", neg_count(m2), "| đáp án =", Solution().maxMatrixSum(m2))
mm = flip(flip(m2, (0, 0), (0, 1)), (0, 1), (1, 1))
print("   sau hai thao tác:", mm, "tổng", sum(map(sum, mm)))
assert sum(map(sum, mm)) == Solution().maxMatrixSum(m2) == 10

# Bài 3
def final_matrix_sol(matrix):
    total, ops = max_sum_constructive(matrix)
    m = [row[:] for row in matrix]
    for a, b in ops:
        m = flip(m, a, b)
    return m

rnd = random.Random(3)
for _ in range(300):
    n = rnd.randint(2, 6)
    mat = [[rnd.randint(-9, 9) for _ in range(n)] for _ in range(n)]
    assert sum(map(sum, final_matrix_sol(mat))) == Solution().maxMatrixSum(mat)
print("Bài 3 OK | ví dụ 2:", final_matrix_sol([[1, 2, 3], [-1, -2, -3], [1, 2, 3]]))

Bài 1: (3, 14)
Bài 2: Σ|a| = 10 | số âm = 2 | đáp án = 10
   sau hai thao tác: [[4, 3], [2, 1]] tổng 10
Bài 3 OK | ví dụ 2: [[1, 2, 3], [1, 2, 3], [-1, 2, 3]]


In [24]:
# Bài 4
def constructive_rect_sol(matrix):
    total, ops = max_sum_constructive(matrix)
    return total, len(ops)

rnd = random.Random(4)
for _ in range(300):
    R, C = rnd.randint(1, 6), rnd.randint(2, 6)
    mat = [[rnd.randint(-9, 9) for _ in range(C)] for _ in range(R)]
    total, k = constructive_rect_sol(mat)
    assert total == max_sum_formula(mat) and k <= 2 * R * C
print("Bài 4 OK")

# Bài 5
def max_sum_row_only_sol(matrix):
    ans = 0
    for row in matrix:
        neg = sum(x < 0 for x in row); s = sum(abs(x) for x in row)
        ans += s if neg % 2 == 0 or len(row) == 0 else s - 2 * min(abs(x) for x in row)
    return ans

def bfs_row_only(matrix):
    R, C = len(matrix), len(matrix[0])
    cells = [(r, c) for r in range(R) for c in range(C)]
    idx = {cell: k for k, cell in enumerate(cells)}
    edges = [(idx[(r, c)], idx[(r, c + 1)]) for r in range(R) for c in range(C - 1)]
    seen = {0}; q = deque([0])
    while q:
        s = q.popleft()
        for i, j in edges:
            t = s ^ (1 << i) ^ (1 << j)
            if t not in seen: seen.add(t); q.append(t)
    return max(sum(-matrix[r][c] if s >> idx[(r, c)] & 1 else matrix[r][c] for r, c in cells) for s in seen)

rnd = random.Random(5)
for _ in range(400):
    R, C = rnd.choice([(2, 2), (2, 3), (3, 3), (3, 2)])
    mat = [[rnd.randint(-5, 5) for _ in range(C)] for _ in range(R)]
    assert max_sum_row_only_sol(mat) == bfs_row_only(mat)
print("Bài 5 OK | [[-1, 2], [3, -4]] ->", max_sum_row_only_sol([[-1, 2], [3, -4]]), "(so với đổi tự do:", Solution().maxMatrixSum([[-1, 2], [3, -4]]), ")")

Bài 4 OK
Bài 5 OK | [[-1, 2], [3, -4]] -> 2 (so với đổi tự do: 10 )


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đổi dấu theo cặp trên đồ thị liên thông":**

```python
total = neg = 0
mn = float('inf')
for x in all_values:
    neg += x < 0
    total += abs(x)
    mn = min(mn, abs(x))
return total if neg % 2 == 0 else total - 2 * mn
```

**Checklist:**
- [ ] Đếm số âm **trước** khi lấy trị tuyệt đối?
- [ ] `min|a|` trên **mọi** phần tử (kể cả dương và 0)?
- [ ] Trừ `2·min|a|` (không phải `min|a|`) khi số âm lẻ?
- [ ] Không chỉ xét cặp âm kề nhau?
- [ ] Đã kiểm bằng BFS trên lưới nhỏ?
- [ ] Chữ ký đúng: `def maxMatrixSum(self, matrix: List[List[int]]) -> int`?